# Lab01 - sprint 2
## Итоговая проверка модели

По validation выбрали kNN с k = 720. Теперь используем весь train, включая бывшую validation.
k не меняем. Полные повторы на границе 720 соседей включаем целиком.
У kNN обучение состоит в сохранении обучающих наблюдений; коэффициенты не подбираются.
Baseline и линейную модель пересчитываем на тех же данных.

Единственный признак - alcohol. Test используем для итоговой оценки, без дальнейшего подбора по его результатам.


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
sys.path.insert(0,str(ROOT/'src'))
from wine_stats.data import load_wines
from wine_stats.split import split_wines
from wine_stats.regression import fit_linear, predict_knn, regression_metrics

K=720
processed=ROOT/'data/processed'
meta=json.loads((processed/'alcohol_quality_split.json').read_text())
for name,info in meta['files'].items():
    assert hashlib.sha256((ROOT/'data/raw'/name).read_bytes()).hexdigest()==info['sha256']
manifest_path=processed/'alcohol_quality_split.csv'
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==meta['manifest_sha256']
manifest=pd.read_csv(manifest_path)
df=load_wines(ROOT/'data/raw')
pd.testing.assert_frame_equal(manifest,split_wines(df,seed=meta['seed'],test_fraction=meta['test_group_fraction']))
assert manifest.groupby('group_id')['split'].nunique().max()==1
train=df.loc[manifest['split'].eq('train')]
test=df.loc[manifest['split'].eq('test')]
print(f'Train: {len(train)}; test: {len(test)}; k = {K}')


Train: 5195; test: 1302; k = 720


In [2]:
baseline=float(train.quality.mean())
a,b=fit_linear(train.alcohol,train.quality)
predictions={
    'mean':np.full(len(test),baseline),
    'linear':a+b*test.alcohol.to_numpy(),
    'knn':predict_knn(train.alcohol,train.quality,test.alcohol,K)}
metrics=pd.DataFrame([{'model':name,'n':len(test),**regression_metrics(test.quality,p)}
                      for name,p in predictions.items()])
display(metrics.round(4))
by_type=[]
for kind in ('red','white'):
    mask=test.wine_type.eq(kind).to_numpy()
    for name,p in predictions.items():
        by_type.append({'wine_type':kind,'model':name,'n':int(mask.sum()),
                        **regression_metrics(test.quality.to_numpy()[mask],p[mask])})
by_type=pd.DataFrame(by_type)
display(by_type.round(4))
tables=ROOT/'reports/tables'
metrics.to_csv(tables/'final_test_metrics.csv',index=False)
by_type.to_csv(tables/'final_test_by_type.csv',index=False)
# Local model artifact: kNN requires the training observations for prediction.
np.savez(processed/'quality_knn_model.npz',alcohol=train.alcohol.to_numpy(),
         quality=train.quality.to_numpy(),k=np.array(K))
saved=np.load(processed/'quality_knn_model.npz',allow_pickle=False)
np.testing.assert_allclose(
    predict_knn(saved['alcohol'],saved['quality'],test.alcohol,int(saved['k'])),
    predictions['knn'])
test_predictions=pd.DataFrame({'row_id':manifest.loc[test.index,'row_id'],
                               'quality':test.quality,**predictions})
test_predictions.to_csv(processed/'final_test_predictions.csv',index=False)
print(f'Линейная модель: quality = {a:.4f} + {b:.4f} × alcohol')


,model,n,MAE,RMSE,R2
0,mean,1302,0.6751,0.8553,-0.0003
1,linear,1302,0.6306,0.7856,0.1560
2,knn,1302,0.6270,0.7770,0.1744


,wine_type,model,n,MAE,RMSE,R2
0,red,mean,314,0.6812,0.8062,-0.0856
1,red,linear,314,0.5926,0.7110,0.1556
2,red,knn,314,0.5801,0.7005,0.1805
3,white,mean,988,0.6732,0.8703,-0.0109
4,white,linear,988,0.6426,0.8079,0.1289
5,white,knn,988,0.6420,0.7998,0.1462


Линейная модель: quality = 2.3132 + 0.3335 × alcohol


## Lab resume

Численные результаты:

In [3]:
result=metrics.set_index('model')
gain=100*(1-result.loc['knn','RMSE']/result.loc['mean','RMSE'])
print(f"kNN: MAE = {result.loc['knn','MAE']:.4f}, RMSE = {result.loc['knn','RMSE']:.4f} балла; R² = {result.loc['knn','R2']:.4f}.")
print(f"Снижение RMSE относительно среднего: {gain:.2f}%.")
print(f"Разность RMSE (kNN − линейная): {result.loc['knn','RMSE']-result.loc['linear','RMSE']:.4f} балла.")


kNN: MAE = 0.6270, RMSE = 0.7770 балла; R² = 0.1744.
Снижение RMSE относительно среднего: 9.15%.
Разность RMSE (kNN − линейная): -0.0086 балла.


По содержанию спирта можно приблизительно предсказать оценку вина дегустатором.
На 1302 тестовых винах модель kNN ошибается в среднем на 0,63 балла, а прогноз одним общим средним - на 0,68 балла.
Учёт спирта улучшил результат, но ненамного: вина одинаковой крепости могут получать разные оценки.
Для более точного прогноза можно исследовать другие характеристики вина, но улучшение нужно проверить отдельно.

Модель прогнозирует числовую оценку дегустаторов. Связь между крепостью и оценкой не доказывает причинное влияние спирта.
Результат относится к этому датасету и не гарантирует такую же ошибку на винах других регионов.
Различия моделей не проверялись на статистическую значимость.

### cat_mark_table

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| - | [Итоговая проверка](03_final_test.ipynb) | - | Отложенный test, MAE/RMSE/R², результаты по типам вина. |
